# Statistical Analysis

In [3]:
import pandas as pd
import numpy as np
import ast
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [4]:
df = pd.read_parquet("../data/processed/anime_data_2.parquet")
df.info()

<class 'pandas.DataFrame'>
Index: 5338 entries, 0 to 8816
Data columns (total 76 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 5338 non-null   int64  
 1   title                  5338 non-null   str    
 2   source                 5338 non-null   str    
 3   episodes               5338 non-null   float64
 4   producers              5338 non-null   object 
 5   genres                 5338 non-null   object 
 6   studios                5338 non-null   object 
 7   demographics           5338 non-null   object 
 8   themes                 5338 non-null   object 
 9   rating                 5338 non-null   str    
 10  sequel                 5338 non-null   bool   
 11  cohort                 5338 non-null   str    
 12  wc_z                   5338 non-null   float64
 13  forum_z                5338 non-null   float64
 14  favorites_z            5338 non-null   float64
 15  score_z             

## Genre, Theme, Demographic vs. Score Z-score (Multiple Regression)

In [134]:
genre_averages = df.explode('genres').groupby('genres')['score_z'].mean().sort_values()
genre_counts = df['genres'].explode().value_counts()
print(genre_counts)
genre_averages

genres
Comedy           2128
Action           1476
Fantasy          1302
Adventure        1191
Sci-Fi            980
Romance           960
Drama             919
Supernatural      518
Mystery           376
Ecchi             334
Slice of Life     305
Sports            276
Suspense          187
Horror            155
Gourmet            78
Award Winning      59
Girls Love         48
Boys Love          37
Avant Garde        35
Erotica            12
Name: count, dtype: int64


genres
Erotica         -0.812554
Ecchi           -0.326011
Avant Garde     -0.231697
Horror          -0.164101
Comedy          -0.116571
Fantasy         -0.052621
Sci-Fi          -0.025447
Adventure        0.038360
Gourmet          0.067773
Action           0.102213
Slice of Life    0.113340
Girls Love       0.114668
Boys Love        0.134078
Romance          0.204279
Supernatural     0.229057
Sports           0.265481
Mystery          0.346265
Drama            0.429990
Suspense         0.479609
Award Winning    1.364233
Name: score_z, dtype: float64

In [135]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['score_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                score_z   R-squared:                       0.113
Model:                            OLS   Adj. R-squared:                  0.111
Method:                 Least Squares   F-statistic:                     40.01
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          3.04e-125
Time:                        18:17:49   Log-Likelihood:                -7127.3
No. Observations:                5338   AIC:                         1.429e+04
Df Residuals:                    5320   BIC:                         1.441e+04
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.3153    

,genre,coef,pval,ci_low,ci_high,pval_fdr,significant
genre_award_winning,genre_award_winning,1.158750,3.631352e-21,0.919200,1.398299,3.086650e-20,True
genre_sports,genre_sports,0.486344,1.191178e-15,0.367609,0.605079,4.050004e-15,True
genre_drama,genre_drama,0.448864,8.832475e-35,0.377846,0.519882,1.501521e-33,True
genre_slice_of_life,genre_slice_of_life,0.372804,4.662310e-10,0.255714,0.489895,1.132275e-09,True
genre_suspense,genre_suspense,0.368882,6.753841e-07,0.223503,0.514260,1.043775e-06,True
genre_romance,genre_romance,0.332028,6.121718e-21,0.262983,0.401073,3.468974e-20,True
genre_mystery,genre_mystery,0.277076,1.527153e-07,0.173737,0.380415,2.884623e-07,True
genre_action,genre_action,0.262716,4.672373e-17,0.201570,0.323863,1.985759e-16,True
genre_supernatural,genre_supernatural,0.251822,6.302210e-08,0.160700,0.342944,1.339220e-07,True
genre_gourmet,genre_gourmet,0.205276,5.261340e-02,-0.002319,0.412871,6.880214e-02,False


In [136]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['score_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                score_z   R-squared:                       0.122
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     19.91
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          1.05e-121
Time:                        18:17:49   Log-Likelihood:                -7101.0
No. Observations:                5338   AIC:                         1.428e+04
Df Residuals:                    5300   BIC:                         1.453e+04
Df Model:                          37                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -0.27

,theme,coef,pval,ci_low,ci_high,pval_fdr,significant
theme_iyashikei,theme_iyashikei,0.876235,6.740423e-17,0.671235,1.081234,8.313188e-16,True
theme_love_polygon,theme_love_polygon,0.576331,2.050659e-07,0.359118,0.793544,9.484297e-07,True
theme_adult_cast,theme_adult_cast,0.555758,2.040119e-22,0.444401,0.667114,7.548439e-21,True
theme_gag_humor,theme_gag_humor,0.547347,2.483440e-11,0.386924,0.707771,2.297182e-10,True
theme_organized_crime,theme_organized_crime,0.511910,9.028600e-05,0.255792,0.768028,2.193310e-04,True
theme_otaku_culture,theme_otaku_culture,0.448778,1.404226e-04,0.217875,0.679682,3.056256e-04,True
theme_urban_fantasy,theme_urban_fantasy,0.444901,3.320117e-07,0.274246,0.615556,1.364937e-06,True
theme_performing_arts,theme_performing_arts,0.441838,2.562097e-04,0.205076,0.678601,5.266532e-04,True
theme_team_sports,theme_team_sports,0.417244,1.362435e-06,0.248101,0.586387,5.041008e-06,True
theme_military,theme_military,0.388398,6.240648e-09,0.257563,0.519232,3.848400e-08,True


In [137]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['score_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                score_z   R-squared:                       0.123
Model:                            OLS   Adj. R-squared:                  0.123
Method:                 Least Squares   F-statistic:                     150.1
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          1.37e-149
Time:                        18:17:49   Log-Likelihood:                -7096.8
No. Observations:                5338   AIC:                         1.421e+04
Df Residuals:                    5332   BIC:                         1.425e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.1503      0.016     -9.358   

,demographic,coef,pval,ci_low,ci_high,pval_fdr,significant
demo_shounen,demo_shounen,0.677593,3.587427e-83,0.610043,0.745142,1.793714e-82,True
demo_shoujo,demo_shoujo,0.587145,3.697029e-20,0.462486,0.711804,4.621286e-20,True
demo_seinen,demo_seinen,0.510448,5.885124e-31,0.424500,0.596395,1.471281e-30,True
demo_josei,demo_josei,0.377501,3.199612e-04,0.171983,0.583018,3.199612e-04,True
demo_kids,demo_kids,-0.551774,6.104430e-30,-0.646355,-0.457193,1.017405e-29,True


## Genre, Theme, Demographic vs. WC Z-score (Multiple Regression)

In [138]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['wc_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                   wc_z   R-squared:                       0.215
Model:                            OLS   Adj. R-squared:                  0.213
Method:                 Least Squares   F-statistic:                     85.94
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          1.73e-264
Time:                        18:17:50   Log-Likelihood:                -6800.7
No. Observations:                5338   AIC:                         1.364e+04
Df Residuals:                    5320   BIC:                         1.376e+04
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.5635    

,genre,coef,pval,ci_low,ci_high,pval_fdr,significant
genre_award_winning,genre_award_winning,0.902129,5.054692e-15,0.676793,1.127464,1.227568e-14,True
genre_romance,genre_romance,0.705622,1.148484e-96,0.640674,0.770570,1.952422e-95,True
genre_suspense,genre_suspense,0.698972,2.004338e-23,0.562220,0.835724,8.518438e-23,True
genre_action,genre_action,0.456861,1.702383e-53,0.399343,0.514379,1.447026e-52,True
genre_sports,genre_sports,0.441491,1.100465e-14,0.329802,0.553181,2.338487e-14,True
genre_ecchi,genre_ecchi,0.367393,6.866049e-13,0.267358,0.467429,1.296920e-12,True
genre_drama,genre_drama,0.360220,7.326992e-26,0.293416,0.427023,4.151962e-25,True
genre_supernatural,genre_supernatural,0.344426,4.022291e-15,0.258711,0.430141,1.139649e-14,True
genre_other,genre_other,0.306254,1.110051e-04,0.151039,0.461469,1.572572e-04,True
genre_fantasy,genre_fantasy,0.286157,1.039264e-19,0.224649,0.347665,3.533498e-19,True


In [139]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['wc_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                   wc_z   R-squared:                       0.203
Model:                            OLS   Adj. R-squared:                  0.197
Method:                 Least Squares   F-statistic:                     36.49
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          2.86e-229
Time:                        18:17:50   Log-Likelihood:                -6842.6
No. Observations:                5338   AIC:                         1.376e+04
Df Residuals:                    5300   BIC:                         1.401e+04
Df Model:                          37                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -0.27

,theme,coef,pval,ci_low,ci_high,pval_fdr,significant
theme_love_polygon,theme_love_polygon,0.769326,3.614233e-13,0.562378,0.976275,1.910380e-12,True
theme_gore,theme_gore,0.718299,1.600749e-15,0.542130,0.894469,9.871284e-15,True
theme_isekai,theme_isekai,0.683643,3.086043e-28,0.562719,0.804567,5.709180e-27,True
theme_psychological,theme_psychological,0.588995,9.320513e-16,0.445743,0.732247,6.897179e-15,True
theme_urban_fantasy,theme_urban_fantasy,0.554843,2.462987e-11,0.392252,0.717433,1.012561e-10,True
theme_harem,theme_harem,0.523114,4.120861e-20,0.411910,0.634318,5.082396e-19,True
theme_otaku_culture,theme_otaku_culture,0.505378,6.825435e-06,0.285386,0.725370,1.803865e-05,True
theme_organized_crime,theme_organized_crime,0.470782,1.571235e-04,0.226767,0.714797,3.229761e-04,True
theme_adult_cast,theme_adult_cast,0.444950,2.498835e-16,0.338855,0.551045,2.311422e-15,True
theme_school,theme_school,0.438449,9.755981e-40,0.373830,0.503068,3.609713e-38,True


In [140]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['wc_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                   wc_z   R-squared:                       0.155
Model:                            OLS   Adj. R-squared:                  0.154
Method:                 Least Squares   F-statistic:                     195.0
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          2.53e-191
Time:                        18:17:50   Log-Likelihood:                -7000.3
No. Observations:                5338   AIC:                         1.401e+04
Df Residuals:                    5332   BIC:                         1.405e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0838      0.016     -5.312   

,demographic,coef,pval,ci_low,ci_high,pval_fdr,significant
demo_shounen,demo_shounen,0.599346,2.982820e-68,0.533007,0.665685,7.457049e-68,True
demo_shoujo,demo_shoujo,0.444651,1.219810e-12,0.322227,0.567076,1.524762e-12,True
demo_seinen,demo_seinen,0.377546,2.394251e-18,0.293139,0.461953,3.990419e-18,True
demo_josei,demo_josei,0.218216,3.409230e-02,0.016382,0.420050,3.409230e-02,True
demo_kids,demo_kids,-0.983189,4.813559e-92,-1.076075,-0.890303,2.406780e-91,True


## Genre, Theme, Demographic vs. Favorites Z-score (Multiple Regression)

In [141]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['favorites_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:            favorites_z   R-squared:                       0.196
Model:                            OLS   Adj. R-squared:                  0.194
Method:                 Least Squares   F-statistic:                     76.39
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          8.60e-237
Time:                        18:17:50   Log-Likelihood:                -6863.3
No. Observations:                5338   AIC:                         1.376e+04
Df Residuals:                    5320   BIC:                         1.388e+04
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.4853    

,genre,coef,pval,ci_low,ci_high,pval_fdr,significant
genre_award_winning,genre_award_winning,1.163331,2.376002e-23,0.935338,1.391325,8.078407e-23,True
genre_suspense,genre_suspense,0.736248,3.116643e-25,0.597883,0.874613,1.324573e-24,True
genre_romance,genre_romance,0.637626,4.179843e-78,0.571912,0.703340,7.105733e-77,True
genre_action,genre_action,0.420404,1.017789e-44,0.362207,0.478600,8.651208e-44,True
genre_drama,genre_drama,0.408394,5.750153e-32,0.340802,0.475986,3.258420e-31,True
genre_sports,genre_sports,0.396886,6.445169e-12,0.283879,0.509893,1.826131e-11,True
genre_other,genre_other,0.363829,5.703478e-06,0.206784,0.520875,9.695913e-06,True
genre_supernatural,genre_supernatural,0.284772,1.324351e-10,0.198045,0.371498,2.814247e-10,True
genre_mystery,genre_mystery,0.260944,2.054302e-07,0.162590,0.359298,3.880348e-07,True
genre_fantasy,genre_fantasy,0.204872,1.188536e-10,0.142639,0.267106,2.814247e-10,True


In [142]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['favorites_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:            favorites_z   R-squared:                       0.184
Model:                            OLS   Adj. R-squared:                  0.179
Method:                 Least Squares   F-statistic:                     32.37
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          2.62e-203
Time:                        18:17:50   Log-Likelihood:                -6902.4
No. Observations:                5338   AIC:                         1.388e+04
Df Residuals:                    5300   BIC:                         1.413e+04
Df Model:                          37                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -0.29

,theme,coef,pval,ci_low,ci_high,pval_fdr,significant
theme_love_polygon,theme_love_polygon,0.885418,1.374458e-16,0.676137,1.094699,1.017099e-15,True
theme_gore,theme_gore,0.712918,5.201365e-15,0.534763,0.891073,3.207508e-14,True
theme_psychological,theme_psychological,0.705953,1.867771e-21,0.561087,0.850820,3.455377e-20,True
theme_isekai,theme_isekai,0.569142,1.007342e-19,0.446856,0.691429,1.242388e-18,True
theme_organized_crime,theme_organized_crime,0.549550,1.290176e-05,0.302786,0.796315,3.672039e-05,True
theme_urban_fantasy,theme_urban_fantasy,0.535938,1.800838e-10,0.371515,0.700361,8.328877e-10,True
theme_iyashikei,theme_iyashikei,0.493836,9.790966e-07,0.296323,0.691349,3.018881e-06,True
theme_adult_cast,theme_adult_cast,0.491272,3.811760e-19,0.383982,0.598562,3.525878e-18,True
theme_otaku_culture,theme_otaku_culture,0.466330,4.028489e-05,0.243859,0.688801,8.767887e-05,True
theme_school,theme_school,0.426671,5.747216e-37,0.361324,0.492018,2.126470e-35,True


In [143]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['favorites_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:            favorites_z   R-squared:                       0.134
Model:                            OLS   Adj. R-squared:                  0.133
Method:                 Least Squares   F-statistic:                     164.6
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          3.02e-163
Time:                        18:17:50   Log-Likelihood:                -7063.1
No. Observations:                5338   AIC:                         1.414e+04
Df Residuals:                    5332   BIC:                         1.418e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0998      0.016     -6.256   

,demographic,coef,pval,ci_low,ci_high,pval_fdr,significant
demo_shounen,demo_shounen,0.595012,7.713787e-66,0.527888,0.662137,3.856893e-65,True
demo_shoujo,demo_shoujo,0.546860,6.470136e-18,0.422986,0.670735,9.991209e-18,True
demo_seinen,demo_seinen,0.375972,7.992967e-18,0.290566,0.461379,9.991209e-18,True
demo_josei,demo_josei,0.295301,4.604355e-03,0.091077,0.499526,4.604355e-03,True
demo_kids,demo_kids,-0.829920,2.317259e-65,-0.923906,-0.735934,5.793148e-65,True


## Genre, Theme, Demographic vs. Drop Rate Z-score (Multiple Regression)

In [5]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['drop_rate_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:            drop_rate_z   R-squared:                       0.138
Model:                            OLS   Adj. R-squared:                  0.135
Method:                 Least Squares   F-statistic:                     49.92
Date:                Wed, 19 Aug 2026   Prob (F-statistic):          1.27e-156
Time:                        23:42:06   Log-Likelihood:                -7053.3
No. Observations:                5338   AIC:                         1.414e+04
Df Residuals:                    5320   BIC:                         1.426e+04
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.3898    

,genre,coef,pval,ci_low,ci_high,pval_fdr,significant
genre_award_winning,genre_award_winning,0.898640,1.029208e-13,0.662387,1.134894,3.499309e-13,True
genre_suspense,genre_suspense,0.794491,3.331335e-27,0.651112,0.937869,1.887757e-26,True
genre_romance,genre_romance,0.515249,8.290990e-49,0.447154,0.583344,1.409468e-47,True
genre_action,genre_action,0.369267,8.926917e-33,0.308961,0.429572,7.587879e-32,True
genre_fantasy,genre_fantasy,0.267097,5.761749e-16,0.202609,0.331585,2.448743e-15,True
genre_ecchi,genre_ecchi,0.197149,2.309761e-04,0.092266,0.302032,4.908243e-04,True
genre_supernatural,genre_supernatural,0.194470,2.250886e-05,0.104602,0.284339,5.466437e-05,True
genre_other,genre_other,0.169858,4.078479e-02,0.007122,0.332594,5.777846e-02,False
genre_mystery,genre_mystery,0.158015,2.381585e-03,0.056097,0.259932,4.498549e-03,True
genre_drama,genre_drama,0.157610,1.047232e-05,0.087569,0.227651,2.967159e-05,True


In [145]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['drop_rate_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:            drop_rate_z   R-squared:                       0.117
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     18.91
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          5.94e-115
Time:                        18:17:50   Log-Likelihood:                -7117.5
No. Observations:                5338   AIC:                         1.431e+04
Df Residuals:                    5300   BIC:                         1.456e+04
Df Model:                          37                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -0.11

,theme,coef,pval,ci_low,ci_high,pval_fdr,significant
theme_love_polygon,theme_love_polygon,0.752053,1.460973e-11,0.534169,0.969937,1.351400e-10,True
theme_gore,theme_gore,0.731905,1.221402e-14,0.546426,0.917384,2.259594e-13,True
theme_psychological,theme_psychological,0.695075,2.261642e-19,0.544253,0.845897,8.368076e-18,True
theme_urban_fantasy,theme_urban_fantasy,0.545522,4.498546e-10,0.374339,0.716704,2.774103e-09,True
theme_isekai,theme_isekai,0.408287,3.497399e-10,0.280973,0.535600,2.588075e-09,True
theme_organized_crime,theme_organized_crime,0.378561,3.883875e-03,0.121653,0.635470,1.105411e-02,True
theme_vampire,theme_vampire,0.307282,4.401179e-03,0.095847,0.518716,1.163169e-02,True
theme_otaku_culture,theme_otaku_culture,0.287915,1.484546e-02,0.056299,0.519532,2.890959e-02,True
theme_adult_cast,theme_adult_cast,0.246907,1.495759e-05,0.135206,0.358607,6.917887e-05,True
theme_harem,theme_harem,0.246673,3.677981e-05,0.129593,0.363753,1.512059e-04,True


In [146]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['drop_rate_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:            drop_rate_z   R-squared:                       0.056
Model:                            OLS   Adj. R-squared:                  0.055
Method:                 Least Squares   F-statistic:                     63.35
Date:                Sun, 16 Aug 2026   Prob (F-statistic):           2.13e-64
Time:                        18:17:50   Log-Likelihood:                -7294.4
No. Observations:                5338   AIC:                         1.460e+04
Df Residuals:                    5332   BIC:                         1.464e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0767      0.017     -4.605   

,demographic,coef,pval,ci_low,ci_high,pval_fdr,significant
demo_shounen,demo_shounen,0.457845,5.389373e-37,0.387748,0.527941,2.694687e-36,True
demo_shoujo,demo_shoujo,0.337238,3.320474e-07,0.207879,0.466597,5.534124e-07,True
demo_seinen,demo_seinen,0.222881,9.914440e-07,0.133693,0.312069,1.239305e-06,True
demo_josei,demo_josei,-0.110539,3.096266e-01,-0.323805,0.102728,3.096266e-01,False
demo_kids,demo_kids,-0.445490,7.643060e-19,-0.543638,-0.347343,1.910765e-18,True


## Genre, Theme, Demographic vs. Forum Z-score (Multiple Regression)

In [147]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['forum_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                forum_z   R-squared:                       0.183
Model:                            OLS   Adj. R-squared:                  0.180
Method:                 Least Squares   F-statistic:                     70.10
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          3.39e-218
Time:                        18:17:50   Log-Likelihood:                -6896.3
No. Observations:                5338   AIC:                         1.383e+04
Df Residuals:                    5320   BIC:                         1.395e+04
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.4767    

,genre,coef,pval,ci_low,ci_high,pval_fdr,significant
genre_award_winning,genre_award_winning,0.773388,4.245692e-11,0.543982,1.002794,1.443535e-10,True
genre_romance,genre_romance,0.609363,7.360397e-71,0.543242,0.675485,1.251267e-69,True
genre_suspense,genre_suspense,0.587731,1.596453e-16,0.448508,0.726954,6.784924e-16,True
genre_drama,genre_drama,0.425586,3.899955e-34,0.357576,0.493597,3.314962e-33,True
genre_action,genre_action,0.363420,1.301608e-33,0.304863,0.421977,7.375778e-33,True
genre_other,genre_other,0.349749,1.457483e-05,0.191731,0.507768,2.064768e-05,True
genre_ecchi,genre_ecchi,0.337922,8.491679e-11,0.236080,0.439765,2.405976e-10,True
genre_mystery,genre_mystery,0.300870,2.683566e-09,0.201906,0.399833,5.068958e-09,True
genre_sports,genre_sports,0.299017,2.623859e-07,0.185310,0.412725,4.055054e-07,True
genre_supernatural,genre_supernatural,0.277430,4.941934e-10,0.190166,0.364694,1.050161e-09,True


In [148]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['forum_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                forum_z   R-squared:                       0.181
Model:                            OLS   Adj. R-squared:                  0.175
Method:                 Least Squares   F-statistic:                     31.56
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          4.20e-198
Time:                        18:17:50   Log-Likelihood:                -6904.4
No. Observations:                5338   AIC:                         1.388e+04
Df Residuals:                    5300   BIC:                         1.413e+04
Df Model:                          37                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -0.27

,theme,coef,pval,ci_low,ci_high,pval_fdr,significant
theme_psychological,theme_psychological,0.712701,8.084602e-22,0.567782,0.857620,1.495651e-20,True
theme_love_polygon,theme_love_polygon,0.667234,4.482394e-10,0.457878,0.876591,1.842762e-09,True
theme_gore,theme_gore,0.579947,1.928516e-10,0.401728,0.758166,1.019358e-09,True
theme_isekai,theme_isekai,0.507709,5.032901e-16,0.385379,0.630040,3.724346e-15,True
theme_adult_cast,theme_adult_cast,0.479709,2.541715e-18,0.372381,0.587038,2.351086e-17,True
theme_urban_fantasy,theme_urban_fantasy,0.460831,4.147472e-08,0.296349,0.625313,1.278804e-07,True
theme_otaku_culture,theme_otaku_culture,0.447450,8.201623e-05,0.224898,0.670001,2.023067e-04,True
theme_military,theme_military,0.440807,8.060929e-12,0.314705,0.566908,4.970906e-11,True
theme_space,theme_space,0.437198,9.811089e-09,0.287961,0.586435,3.300094e-08,True
theme_school,theme_school,0.407344,7.270394e-34,0.341973,0.472714,2.690046e-32,True


In [149]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['forum_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

                            OLS Regression Results                            
Dep. Variable:                forum_z   R-squared:                       0.149
Model:                            OLS   Adj. R-squared:                  0.148
Method:                 Least Squares   F-statistic:                     186.3
Date:                Sun, 16 Aug 2026   Prob (F-statistic):          2.41e-183
Time:                        18:17:50   Log-Likelihood:                -7006.1
No. Observations:                5338   AIC:                         1.402e+04
Df Residuals:                    5332   BIC:                         1.406e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0440      0.016     -2.788   

,demographic,coef,pval,ci_low,ci_high,pval_fdr,significant
demo_shounen,demo_shounen,0.431832,1.102362e-36,0.365421,0.498244,2.755906e-36,True
demo_seinen,demo_seinen,0.401770,1.646340e-20,0.317271,0.486270,2.743900e-20,True
demo_shoujo,demo_shoujo,0.331712,1.166214e-07,0.209153,0.454270,1.457767e-07,True
demo_josei,demo_josei,0.221953,3.132678e-02,0.019898,0.424008,3.132678e-02,True
demo_kids,demo_kids,-1.105615,1.611544e-114,-1.198603,-1.012627,8.057721e-114,True


## Rating, Sequel vs. Score Z-score (ANOVA + Tukey, T-test)

In [150]:
# one-way ANOVA
groups = [df.loc[df['rating'] == r, 'score_z'].dropna() for r in df['rating'].unique()]
f_stat, p_val = stats.f_oneway(*groups)
print(f"F={f_stat:.3f}, p={p_val:.4f}")

# Tukey post-hoc
tukey = pairwise_tukeyhsd(endog=df['score_z'], groups=df['rating'], alpha=0.05)
print(tukey)

F=112.488, p=0.0000
                        Multiple Comparison of Means - Tukey HSD, FWER=0.05                         
            group1                         group2             meandiff p-adj   lower   upper  reject
----------------------------------------------------------------------------------------------------
                  G - All Ages                  PG - Children  -0.1086 0.3128 -0.2639  0.0467  False
                  G - All Ages      PG-13 - Teens 13 or older   0.4476    0.0  0.3447  0.5504   True
                  G - All Ages R - 17+ (violence & profanity)   0.8563    0.0  0.7184  0.9942   True
                  G - All Ages               R+ - Mild Nudity    0.065 0.8509 -0.1106  0.2406  False
                 PG - Children      PG-13 - Teens 13 or older   0.5562    0.0  0.4235  0.6889   True
                 PG - Children R - 17+ (violence & profanity)   0.9649    0.0  0.8035  1.1263   True
                 PG - Children               R+ - Mild Nudity   0.1736 

In [151]:
seq_yes = df.loc[df['sequel'] == 1, 'score_z'].dropna()
seq_no  = df.loc[df['sequel'] == 0, 'score_z'].dropna()

t_stat, p_val = stats.ttest_ind(seq_yes, seq_no, equal_var=False)  # Welch's, safer default
print(f"t={t_stat:.3f}, p={p_val:.4f}, mean_diff={seq_yes.mean() - seq_no.mean():.3f}")

t=11.515, p=0.0000, mean_diff=0.361


We want to also find the partial effects of ratings and sequels taking into account genres, themes, and demographics since they are likely related (e.g. demographics and age ratings definitely correlate). By "partial effects," I mean that if we keep everything else constant, what effect does it have on the score z-score?

In [152]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['sequel']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['score_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                score_z   R-squared:                       0.312
Model:                            OLS   Adj. R-squared:                  0.304
Method:                 Least Squares   F-statistic:                     42.76
Date:                Sun, 16 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:17:51   Log-Likelihood:                -6449.2
No. Observations:                5338   AIC:                         1.303e+04
Df Residuals:                    5273   BIC:                         1.346e+04
Df Model:                          64                                         
Covariance Type:                  HC3                                         
                                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

In [153]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

genre_award_winning                      1.074797
theme_iyashikei                          0.768610
demo_shounen                             0.598384
theme_gag_humor                          0.485374
theme_otaku_culture                      0.423685
demo_shoujo                              0.418721
theme_mahou_shoujo                       0.400001
theme_adult_cast                         0.390427
genre_drama                              0.390363
rating_R - 17+ (violence & profanity)    0.384240
Name: coef, dtype: float64
theme_military                      0.155622
theme_mecha                         0.149332
theme_other                         0.102994
genre_action                        0.090077
rating_PG-13 - Teens 13 or older    0.089288
theme_strategy_game                -0.158327
theme_harem                        -0.188629
genre_ecchi                        -0.241679
demo_kids                          -0.256844
genre_horror                       -0.375300
Name: coef, dtype: floa

The above shows the top 10 and bottom 10 coefficients for features that yielded statistically significant results $(p < 0.05)$.

## Other Metrics

For the other metrics, we will go straight to partial effects.

In [154]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['sequel']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['wc_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   wc_z   R-squared:                       0.419
Model:                            OLS   Adj. R-squared:                  0.412
Method:                 Least Squares   F-statistic:                     69.32
Date:                Sun, 16 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:17:51   Log-Likelihood:                -5998.0
No. Observations:                5338   AIC:                         1.213e+04
Df Residuals:                    5273   BIC:                         1.255e+04
Df Model:                          64                                         
Covariance Type:                  HC3                                         
                                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

In [155]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

genre_award_winning                      0.780794
rating_R - 17+ (violence & profanity)    0.716094
theme_otaku_culture                      0.500328
demo_shounen                             0.476691
theme_iyashikei                          0.464410
theme_isekai                             0.437054
theme_love_polygon                       0.388165
rating_R+ - Mild Nudity                  0.385825
genre_romance                            0.383997
rating_PG-13 - Teens 13 or older         0.383752
Name: coef, dtype: float64
genre_other            0.146844
theme_mecha            0.141556
theme_super_power      0.131495
theme_other            0.114457
theme_parody          -0.177288
genre_horror          -0.177430
theme_martial_arts    -0.225836
theme_samurai         -0.270150
theme_strategy_game   -0.318106
demo_kids             -0.325816
Name: coef, dtype: float64


In [156]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['sequel']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['favorites_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            favorites_z   R-squared:                       0.385
Model:                            OLS   Adj. R-squared:                  0.378
Method:                 Least Squares   F-statistic:                     60.02
Date:                Sun, 16 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:17:51   Log-Likelihood:                -6147.1
No. Observations:                5338   AIC:                         1.242e+04
Df Residuals:                    5273   BIC:                         1.285e+04
Df Model:                          64                                         
Covariance Type:                  HC3                                         
                                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

In [157]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

genre_award_winning                      1.024755
rating_R - 17+ (violence & profanity)    0.685179
theme_iyashikei                          0.628964
demo_shounen                             0.510101
theme_love_polygon                       0.503539
theme_otaku_culture                      0.487852
theme_isekai                             0.383777
theme_mahou_shoujo                       0.381767
demo_shoujo                              0.376007
theme_gore                               0.366633
Name: coef, dtype: float64
genre_fantasy          0.186174
theme_super_power      0.129063
theme_mecha            0.120504
theme_other            0.104804
sequel                -0.106048
demo_kids             -0.214185
theme_detective       -0.228623
theme_samurai         -0.235981
genre_horror          -0.282922
theme_strategy_game   -0.290967
Name: coef, dtype: float64


In [158]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['sequel']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['drop_rate_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            drop_rate_z   R-squared:                       0.257
Model:                            OLS   Adj. R-squared:                  0.248
Method:                 Least Squares   F-statistic:                     27.79
Date:                Sun, 16 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:17:51   Log-Likelihood:                -6655.1
No. Observations:                5338   AIC:                         1.344e+04
Df Residuals:                    5273   BIC:                         1.387e+04
Df Model:                          64                                         
Covariance Type:                  HC3                                         
                                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

In [159]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

genre_award_winning                      0.783274
theme_love_polygon                       0.526465
theme_gore                               0.467623
demo_shounen                             0.405482
rating_R - 17+ (violence & profanity)    0.402549
theme_psychological                      0.378825
theme_otaku_culture                      0.320602
genre_romance                            0.309139
genre_action                             0.305905
theme_urban_fantasy                      0.291567
Name: coef, dtype: float64
rating_PG-13 - Teens 13 or older    0.200851
genre_adventure                     0.171460
genre_supernatural                  0.149490
genre_drama                         0.120077
demo_seinen                         0.107411
theme_historical                   -0.115307
theme_music                        -0.148612
demo_josei                         -0.199067
theme_strategy_game                -0.208382
sequel                             -0.453779
Name: coef, dtype: floa

In [160]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['sequel']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['forum_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                forum_z   R-squared:                       0.392
Model:                            OLS   Adj. R-squared:                  0.385
Method:                 Least Squares   F-statistic:                     58.72
Date:                Sun, 16 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:17:52   Log-Likelihood:                -6107.3
No. Observations:                5338   AIC:                         1.234e+04
Df Residuals:                    5273   BIC:                         1.277e+04
Df Model:                          64                                         
Covariance Type:                  HC3                                         
                                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

In [161]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

rating_R - 17+ (violence & profanity)    0.780008
genre_award_winning                      0.592356
rating_PG-13 - Teens 13 or older         0.485001
rating_R+ - Mild Nudity                  0.465697
theme_otaku_culture                      0.431052
theme_iyashikei                          0.421501
theme_cgdct                              0.420974
theme_mahou_shoujo                       0.388882
demo_shounen                             0.369987
theme_isekai                             0.331714
Name: coef, dtype: float64
genre_other             0.173249
genre_ecchi             0.170231
theme_historical        0.117533
theme_military          0.116115
genre_sci-fi            0.114320
theme_other             0.098395
sequel                 -0.133295
rating_PG - Children   -0.169228
demo_kids              -0.298705
theme_strategy_game    -0.426843
Name: coef, dtype: float64
